In [2]:
from phase_II.nifty_re_playground.useful.helpers import *
%matplotlib tk

In [11]:
def tukey_window_matrix(mat, ax=None, alpha=0.2):
    ny, nx = mat.shape

    if ax is None:
        # Window both axes
        wx = tukey(nx, alpha=alpha)
        wy = tukey(ny, alpha=alpha)
        window2d = wy[:, None] * wx[None, :]
    elif ax == 0 or ax == 1:
        # Window only rows (ax=0) or columns (ax=1)
        if ax == 0:
            wy = tukey(ny, alpha=alpha)
            window2d = wy[:, None]  # broadcast along columns
        else:  # ax==1
            wx = tukey(nx, alpha=alpha)
            window2d = wx[None, :]  # broadcast along rows
    else:
        raise ValueError("ax must be None, 0 (rows), or 1 (columns)")

    return mat * window2d

def tukey_window_array(array, alpha=0.2):
    wx = tukey(len(array), alpha=alpha)
    return array * wx

In [75]:
def Stress_re_debug(xi, time, supress_print=False, downsample=False, norm="ortho", tukey_window_where_necessary=False):
    """
    Implements S_ft, i.e. rows are frequencies and columns are times.

    See also nifty8 `Stress` function.

    :param xi: jnp.array        A field to calculate the wigner function for. Either of complex or real data type.
                                If complex, assumed to be in DFT standard order (DC first, then positives then negatives).
    :param time: jnp.array      The real-space time array at which xi (or its iFFT if complex) was sampled at.
    :param supress_print: bool, Print imaginary part diagonstics (Wigner function should be real).
    :return:
    """

    t0 = time[0]
    dt = time[1]-time[0]
    N = len(xi)
    f = jnp.fft.fftfreq(N, d=dt)
    k = f.copy()
    df = f[1] - f[0]
    t = jnp.arange(N) / (N*df)  # dual time, equal to input time - time[0].
    T = N * dt

    FFT_physical = lambda x, ax=-1: jnp.fft.fft(x, norm=norm, axis=ax) * T / jnp.sqrt(N)
    iFFT_physical = lambda x, ax=-1: jnp.fft.ifft(x, norm=norm, axis=ax) * jnp.sqrt(N) / T

    if jnp.iscomplexobj(xi):
        xi = iFFT_physical(xi)  # go to real space

    if downsample:
        step = 2
        xi = xi[::step]
        time = time[::step]

    if not supress_print:
        print("\nCalculating stress...")

    t_c = t[:, None]  # time cast
    k_c = k[None, :]  # shift frequencies cast
    xi_c = xi[:, None]  # xi values cast as rows

    if not supress_print:
        print("\t Calculating zeta plus")
    if tukey_window_where_necessary:
        zeta_plus = tukey_window_matrix(jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c, ax=0) # domain = (time_space, h_space)
    else:
        zeta_plus = jnp.exp(-jnp.pi * k_c * 1j * t_c) * xi_c # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta minus")

    if tukey_window_where_necessary:
        zeta_minus = tukey_window_matrix(jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c, ax=0) # domain = (time_space, h_space)
    else:
        zeta_minus = jnp.exp(jnp.pi * k_c * 1j * t_c) * xi_c  # domain = (time_space, h_space)

    if not supress_print:
        print("\t Calculating zeta plus in Fourier space")
    tilde_zeta_plus = FFT_physical(zeta_plus, ax=0)

    if not supress_print:
        print("\t Calculating zeta minus in Fourier space")
    tilde_zeta_minus = FFT_physical(zeta_minus, ax=0)

    if not supress_print:
        print("\t Calculating Phi matrix")
    if tukey_window_where_necessary:
        Phi = tukey_window_matrix(tilde_zeta_plus * tilde_zeta_minus.conj(), ax=1)  # domain = (h_space, h_space)
    else:
        Phi = tilde_zeta_plus * tilde_zeta_minus.conj()  # domain = (h_space, h_space)

    if not supress_print:
        print("\t Inverse Fourier-Transforming columns of Phi matrix")
    S = iFFT_physical(Phi, ax=1)
    S.block_until_ready()

    if not supress_print:
        print("\t ... Done")
    if not supress_print:
        diagnostic = jnp.abs(jnp.mean(S.imag))
        tmp = float(diagnostic)
        if diagnostic < 1e-10:
            print(f"\u2714 Mean imaginary part of stress field is smaller than 1e-10 threshold ({diagnostic}) ")
        else:
            raise_warning(
                f"Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 ({diagnostic}).")
    return S, t+t0, f

In [5]:
def generate_white_noise_stress_matrices(number_of_matrices, time_array, std=1, supress_print=False, complex=False, norm="ortho"):
    """

    :param number_of_matrices:  How many matrices to generate
    :param time_array:         The array containing the time samples
    :param std:                 Is multiplied onto xi iid. For example, if you want to represent a Dirac Delta, the covariance should be Kronecker Delta * T,
                                where T is the length of the time domain. Therefore, variables have to be generated that are scaled with the standard deviation sqrt(T)
    :param supress_print:
    :return:
    """
    S_mat_collection = []
    N = len(time_array)
    for i in range(number_of_matrices):
            print(f"Calcuting white noise stress matrix, iteration {i} out of {number_of_matrices}")
            if complex:
                 white_noise = 1/2*np.random.randn(N) + 1/2*np.random.randn(N) * 1j  # I have not calculated the variance or mean for the complex fields, therefore
                 # I cannot benchmark this
            else:
                white_noise = np.random.standard_normal(N)
            white_noise_scaled = std * white_noise
            stress, _, _ = Stress_re_debug(white_noise_scaled, time=time_array, supress_print=supress_print, norm=norm)
            S_mat_collection.append(np.array(stress))

    print("Done, wrapping result in numpy array")
    return np.array(S_mat_collection)

### White noise Wigner function for real and complex field at different resolutions; pure white noise matrix

In [12]:
res_1 = 256
res_2 = 256*10
T = 3
time_res1 = jnp.linspace(0, T, res_1)
time_res2 = jnp.linspace(0, T, res_2)

delta_t_1 = time_res1[1]-time_res1[0]
delta_t_2 = time_res2[1]-time_res2[0]

xi_white_complex_res1 = np.random.standard_normal(res_1) + 1j*np.random.standard_normal(res_1)

xi_white_real_res1 = np.random.standard_normal(res_1)
xi_white_real_res2 = np.random.standard_normal(res_2)


In [5]:
white_stress_complex_res1, t_dual_res_1, f_dual_res_1 = Stress_re_debug(xi=xi_white_complex_res1, time=time_res1)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done






/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:1513: UserWarning: Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 (7.171365723479539e-10).
  raise_warning(


In [6]:
white_stress_real_res1, t_dual_res_1, f_dual_res_1 = Stress_re_debug(xi=xi_white_real_res1, time=time_res1)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.2878587085651816e-11) 


In [22]:
white_stress_real_res2, t_dual_res_2, f_dual_res_2 = Stress_re_debug(xi=xi_white_real_res2, time=time_res2)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (4.924553671206873e-14) 


In [8]:
pure_white_noise_res1 = np.random.standard_normal(white_stress_real_res1.shape)
pure_white_noise_res2 = np.random.standard_normal(white_stress_real_res2.shape)

In [23]:
# Visualize
apply_smoothing=True
save_current_figure=False

# fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True)

# flattened_axs = axs.flatten()

# visualize_stress(white_stress_complex_res1, rows=f_dual_res_1, cols=t_dual_res_1, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Wigner function: complex $\xi$, low resolution")
# visualize_stress(white_stress_real_res1, rows=f_dual_res_1, cols=t_dual_res_1, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Wigner function: real $\xi$, low resolution")
visualize_stress(white_stress_real_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Wigner function: real $\xi$, high resolution")
# visualize_stress(pure_white_noise_res1, rows=f_dual_res_1, cols=t_dual_res_1, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Pure white noise, low resolution")
# visualize_stress(pure_white_noise_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=apply_smoothing, save_fig=save_current_figure, tl=r"Pure white noise, smoothed", show=False)
# visualize_stress(pure_white_noise_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=not apply_smoothing, save_fig=save_current_figure, tl=r"Pure white noise, smoothed", show=False, yl=None)
plt.show()

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:73: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [13]:
visualize_stress(white_stress_real_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=True, save_fig=False, tl=r"Wigner function: real $\xi$, high resolution")

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


In [ ]:
pnum_of_samples = 10
white_noise_stress_collection = generate_white_noise_stress_matrices(number_of_matrices=num_of_samples, time_array=time_res2, supress_print=True, std=1/np.sqrt(delta_t_2),
                                                                     complex=False, norm="ortho")

In [ ]:
white_noise_stress_average = np.mean(white_noise_stress_collection, axis=0)

In [ ]:
print(np.mean(white_noise_stress_average), "should be 1")
print(np.std(white_noise_stress_average), f"should be {jnp.sqrt(res_2)/jnp.sqrt(num_of_samples)} = sqrt(delta(0) in time * delta(0) in frequency) / sqrt(num_of_samples) ")

In [ ]:
visualize_stress(white_noise_stress_average, rows=f_dual_res_1, cols=t_dual_res_1
                 )

In [ ]:
res = res_2
place_to_place_differing_gaussian_matrix = np.zeros((res, res))
for i in range(res):
    for j in range(res):
        rnd_sigma = np.abs(2*np.random.standard_normal(1)+.2)
        # rnd_sigma = 2
        rnd_mean  = np.abs(2*np.random.standard_normal(1)+2)
        # rnd_mean  = 1
        gaussian_sample = rnd_mean + rnd_sigma*np.random.standard_normal(size=1)
        place_to_place_differing_gaussian_matrix[i, j] = gaussian_sample[0]

In [ ]:
res = res_2
place_to_place_differing_lognormal_matrix = np.zeros((res, res))
for i in range(res):
    for j in range(res):
        rnd_sigma = np.abs(2*np.random.standard_normal(1)+.2)
        rnd_mean  = np.abs(2*np.random.standard_normal(1)+2)
        ln_sample = np.random.lognormal(rnd_mean, rnd_sigma, size=1)[0]
        place_to_place_differing_lognormal_matrix[i, j] = ln_sample

In [ ]:
res = res_2
lognormal_matrix = np.zeros((res, res))
for i in range(res):
    for j in range(res):
        ln_sample = np.random.lognormal(mean=1, sigma=0.1, size=1)[0]
        lognormal_matrix[i, j] = ln_sample

In [ ]:
# visualize_stress(place_to_place_differing_gaussian_matrix, rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)
visualize_stress(pure_white_noise_res2, rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)

In [ ]:
visualize_stress(np.log(place_to_place_differing_lognormal_matrix), rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)

In [ ]:
visualize_stress(np.log(lognormal_matrix), rows=f_dual_res_2, cols=t_dual_res_2, smooth=True)

In [ ]:
plt.hist(np.sum(place_to_place_differing_gaussian_matrix.real, axis=0), bins=100)
plt.show()

In [15]:
visualize_stress(white_stress_real_res1, rows=f_dual_res_1, cols=t_dual_res_1, smooth=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:77: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


## Numerical relativity template

In [27]:
from phase_I.utils.config_jupyter_notebooks import *
nrt_strain_values = jnp.array(np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_strain_values.txt") * 1e19)
nrt_time_values = jnp.array(np.loadtxt("/Users/iason/PycharmProjects/STRAIN/data/data_txt/num_rel_template_time_values.txt") - zero_time)

In [28]:
S_mat_nrt, t_dual_nrt, f_nrt = Stress_re_debug(xi=nrt_strain_values, time=nrt_time_values)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (1.9181340264030368e-18) 


In [29]:
visualize_stress(S_mat_nrt, rows=f_nrt, cols=t_dual_nrt, smooth=False)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:73: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [41]:
import jax.numpy as jnp
from phase_II.nifty_re_playground.useful.helpers import *
from phase_II.nifty_re_playground.useful.helpers import convert_gps_to_seconds
from phase_II.nifty_re_playground.useful.basics.welch_average import calculate_welch_average
from phase_II.utils.helpers import whiten, bandpass

def get_xi_for_event(event_idx=3, off_center=1.5, abs_path_to_use=None):
    # Define events
    events = [
        {"gw_name": 'GW150914', "duration":32, "unpack":True, "version": 3, "sample_rate":4096, "center_at": 1126259462.4+off_center},
        {"gw_name": 'GW150914', "duration":4096, "unpack":True, "version": 4, "sample_rate":4096, "center_at": 1126259598-4+off_center},
        {"gw_name": 'GW250114_082203', "duration":32, "unpack":True, "version": 2, "sample_rate":4, "center_at": 1420878141.2+off_center},
        {"gw_name": 'GW190521_074359', "duration":32, "unpack":True, "version": 2, "sample_rate":4, "center_at": 1242459857.4+off_center},
    ]
    event = events[event_idx]

    # Load strain data
    times, strain, t0_gps = get_strain_data(**event, absolute_path=abs_path_to_use)

    # Compute marker to select the window
    marker = convert_gps_to_seconds(event["center_at"]-off_center, t0=t0_gps)

    # Compute power spectral density using Welch averaging
    f, ps, windows = calculate_welch_average(x=times, y=strain, L=2)

    # Loop through windows to find the one containing the event
    for current_window in windows:
        current_time, current_strain = current_window
        if current_time.min() < marker < current_time.max():
            # Whiten the strain in this window
            my_whitened = whiten(y=current_strain, amp=jnp.sqrt(ps))
            # Optionally bandpass (comment out if not needed)
            my_bandpassed = bandpass(x=current_time, y=my_whitened)
            # Solve for xi in frequency space
            current_strain_windowed = tukey_window_array(current_strain)
            xi_d_tilde = solve_data_equation_for_xi(data=current_strain_windowed, ps=ps)
            return xi_d_tilde, current_time, f

    raise ValueError("No window contains the event marker.")


In [94]:
abs_path_to_use = "/Users/iason/PycharmProjects/STRAIN/data/data_pickle_or_hdf5/gwpy_objects/H-H1_GWOSC_4KHZ_R1-1242459842-32.hdf5"  # idx=3
# abs_path_to_use = "/Users/iason/PycharmProjects/STRAIN/data/data_pickle_or_hdf5/gwpy_objects/H-H1_LOSC_4_V1-1126256640-4096.hdf5"  # idx=0
xi_d_tilde, times_window, freqs = get_xi_for_event(event_idx=3, abs_path_to_use=abs_path_to_use)
print("xi shape:", xi_d_tilde.shape)
print("time array shape:", times_window.shape)


hey=? 
Start: Calculating welch average

Constructing 16 windows over which we average.

Done
xi shape: (8193,)
time array shape: (8193,)


In [95]:
S_mat_inference, t_dual_inference, f_inference = Stress_re_debug(xi=xi_d_tilde, time=times_window, tukey_window_where_necessary=False)
S_mat_inference_tukey, t_dual_inference_tukey, f_inference_tukey = Stress_re_debug(xi=jnp.fft.ifft(xi_d_tilde).real, time=times_window, tukey_window_where_necessary=True)


Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix
	 ... Done





Calculating stress...
	 Calculating zeta plus
	 Calculating zeta minus
	 Calculating zeta plus in Fourier space
	 Calculating zeta minus in Fourier space
	 Calculating Phi matrix
	 Inverse Fourier-Transforming columns of Phi matrix


/var/folders/6s/zt639pwd3kg46kcb8b3t121r0000gp/T/ipykernel_68221/2550641725.py:84: UserWarning: Realness threshold was not passed. Mean imaginary part of stress field larger than 1e-10 (2.9612341677420773e-05).
  raise_warning(


	 ... Done
✔ Mean imaginary part of stress field is smaller than 1e-10 threshold (3.568852299188491e-13) 


In [96]:
S_mat_inference_shifted = np.fft.fftshift(S_mat_inference)
S_mat_inference_tukey_shifted = np.fft.fftshift(S_mat_inference_tukey)

In [117]:
# visualize_stress(S_mat_inference, rows=f_inference, cols=t_dual_inference, smooth=True)
visualize_stress(S_mat_inference_tukey, rows=f_inference_tukey, cols=t_dual_inference_tukey, smooth=True)

		Rows must be increasing, assuming a priori standard DFT order and moving DC to the middle


/Users/iason/PycharmProjects/STRAIN/phase_II/nifty_re_playground/useful/helpers.py:73: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [113]:
import numpy as np


def plot_middle_slice(mat, time, smoothing_lvl=None, abs_square=False, title=None, normalize=False, plot=True):
    """
    Plots the middle slice of a 2D matrix along the first axis (frequency-like axis).

    :param mat: 2D array, shape (n_f, n_t)
    :param time: 1D array, length n_t, corresponding to the second axis of mat
    :param smoothing_lvl: int or None, if given applies Gaussian smoothing via smooth_matrix
    :param abs_square: bool, if True, takes the squared absolute value of the matrix
    :param title: optional plot title
    """
    import copy
    mat_plot = copy.deepcopy(mat)

    # Optional smoothing
    if smoothing_lvl is not None:
        mat_plot = smooth_matrix(mat_plot, smoothing_lvl=smoothing_lvl, mode="gaussian")

    # Optional absolute square
    if abs_square:
        mat_plot = np.array(mat_plot.copy().real**2)

    # Take middle slice along first axis
    middle_idx = mat_plot.shape[0] // 2
    slice_middle = mat_plot[middle_idx, :]

    if normalize:
        slice_middle /= np.max(slice_middle)
        print("avg: ", np.average(slice_middle))

    if plot:
        # Plot
        plt.figure(figsize=(8,4))
        plt.plot(time, slice_middle)
        plt.xlabel("Time")
        plt.ylabel("Amplitude")
        if title is not None:
            plt.title(title)
        plt.grid(True)
        plt.show()
    return slice_middle



In [115]:
slice_middle_1 = plot_middle_slice(S_mat_inference_shifted, times_window, smoothing_lvl=3, abs_square=False, normalize=True, plot=True)
slice_middle_2 = plot_middle_slice(S_mat_inference_tukey_shifted, times_window, smoothing_lvl=3, abs_square=True, normalize=True, plot=True)

avg:  (9.409272e-05+9.0982727e-10j)


/Users/iason/PycharmProjects/stability-of-submoons/.venv/lib/python3.12/site-packages/matplotlib/cbook.py:1719: ComplexWarning: Casting complex values to real discards the imaginary part
  return math.isfinite(val)
/Users/iason/PycharmProjects/stability-of-submoons/.venv/lib/python3.12/site-packages/matplotlib/cbook.py:1355: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)


avg:  0.0113847535


In [112]:
# Select the time range
mask = (times_window < 15.5) | (times_window > 15.63)

# Apply mask to slices and time
t_sel = times_window[mask]
s1_sel = slice_middle_1[mask]
s2_sel = slice_middle_2[mask]

# Plot both slices
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.plot(t_sel, s1_sel, label="Slice 1")
plt.plot(t_sel, s2_sel, label="Slice 2")
plt.xlabel("Time")
plt.ylabel("Amplitude")
plt.title("Slices between 15.5 and 15.63")
plt.legend()
plt.grid(True)
plt.show()

# Compute and print mean values
print("Mean of Slice 1 in range:", np.mean(s1_sel))
print("Mean of Slice 2 in range:", np.mean(s2_sel))


Mean of Slice 1 in range: 0.012213861
Mean of Slice 2 in range: 0.007109558
